## Reasoning Patch

In [1]:
%load_ext autoreload
%autoreload 2

### Overview

Patches several token positions at once, within a single layer, to test whether jointly intervening on multiple addend tokens have different effects than intervening on them individually through the `targeted.ipynb` file.

### Set-up

In [ ]:
import torch
import gc
from tqdm import tqdm

import sys
sys.path.append("src")
import _config
import _util
from _intervention import prepare_batch_multitoken_intervention, batch_intervene, get_attention_freeze_hooks

In [3]:
_util.print_GPU_availbility()

CUDA is available: True
Available devices:
  GPU 0: NVIDIA RTX A5500
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |      0 B   |
|       from small pool |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |      0 B   |      0 B   |      0 B

In [4]:
prompt_config = _config.PromptConfig(
    model_type="GPT-OSS", # GPT-OSS or R1
    prompt_type="h1_pre_result", # empty or pre_result or pre_final_sum or h or h_pre_result or h_pre_final_sum
)
run_config = _config.RunConfig(
    experiment_root="experiments/activation_intervention",
    output_filename=f"restatement_patching_layer_{{layer}}{prompt_config.suffix}.csv",
)
model_type = prompt_config.model_type
prompt_type = prompt_config.suffix

model, tokenizer = _config.load_model(prompt_config.model_type)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [5]:
prompts = _config.load_prompts(prompt_config)
print(f"loaded {len(prompts)} prompts")

loaded 256 prompts


In [6]:
for i, row in prompts.iterrows():
    base_prompt = row['base_prompt']
    source_prompt = row['source_prompt']
    base_prompt = tokenizer(base_prompt, add_special_tokens=False, return_tensors="pt")["input_ids"][0]
    source_prompt = tokenizer(source_prompt, add_special_tokens=False, return_tensors="pt")["input_ids"][0]
    base_prompt_len = base_prompt.shape[0]
    source_prompt_len = source_prompt.shape[0]
    print(f"base_prompt_len: {base_prompt_len}, source_prompt_len: {source_prompt_len}")
    print(list(enumerate(tokenizer.convert_ids_to_tokens(base_prompt))))
    print(list(enumerate(tokenizer.convert_ids_to_tokens(source_prompt))))

# [112, 202, 212]
# [110, 112, 199, 202, 209, 212]
    

base_prompt_len: 258, source_prompt_len: 258
[(0, '<|start|>'), (1, 'system'), (2, '<|message|>'), (3, 'You'), (4, 'Ġare'), (5, 'ĠChat'), (6, 'GPT'), (7, ','), (8, 'Ġa'), (9, 'Ġlarge'), (10, 'Ġlanguage'), (11, 'Ġmodel'), (12, 'Ġtrained'), (13, 'Ġby'), (14, 'ĠOpen'), (15, 'AI'), (16, '.Ċ'), (17, 'Knowledge'), (18, 'Ġcutoff'), (19, ':'), (20, 'Ġ'), (21, '202'), (22, '4'), (23, '-'), (24, '06'), (25, 'Ċ'), (26, 'Current'), (27, 'Ġdate'), (28, ':'), (29, 'Ġ'), (30, '202'), (31, '5'), (32, '-'), (33, '06'), (34, '-'), (35, '28'), (36, 'ĊĊ'), (37, 'Reason'), (38, 'ing'), (39, ':'), (40, 'Ġhigh'), (41, 'Ċ'), (42, '=Ċ'), (43, '#'), (44, 'ĠValid'), (45, 'Ġchannels'), (46, ':'), (47, 'Ġanalysis'), (48, ','), (49, 'Ġcommentary'), (50, ','), (51, 'Ġfinal'), (52, '.'), (53, 'ĠChannel'), (54, 'Ġmust'), (55, 'Ġbe'), (56, 'Ġincluded'), (57, 'Ġfor'), (58, 'Ġevery'), (59, 'Ġmessage'), (60, '.'), (61, '<|end|>'), (62, '<|start|>'), (63, 'developer'), (64, '<|message|>'), (65, '#'), (66, 'ĠInstruction'), 

## Patching

Patches the residual stream at a single fixed layer (`LAYER=16`) across all addends specified in the tok_pos_list.

In [ ]:
LAYER = 16
tok_pos_list = [110, 112, 199, 202, 209, 212]
run_config = _config.RunConfig(
    experiment_root="experiments/activation_intervention",
    output_filename=f"restatement_patching_layer_{LAYER}{prompt_config.suffix}.csv",
)

header = list(prompts.columns) + ['generated_text', 'intervention_id']
filepath = _config.build_run_output_filepath(prompt_config, run_config, header)

batch_size = 24
df = prompts
# df = divided_prompts[divided_prompts['intervention_id'] == 25]

for i in tqdm(range(0, len(df), batch_size)):
    torch.cuda.empty_cache()
    gc.collect()
    batch_rows = df.iloc[i:i+batch_size]

    tokens, _, hook = prepare_batch_multitoken_intervention(model, tokenizer, LAYER, tok_pos_list, batch_rows['base_prompt'].tolist(), batch_rows['source_prompt'].tolist())
    input_length = tokens["input_ids"].shape[1]

    for j in range(3):
        # attention_freeze_hooks = get_attention_freeze_hooks(model, tokens)
        with torch.no_grad():
            output = batch_intervene(model, tokens["input_ids"], [hook], attention_mask=tokens["attention_mask"]) # attention_freeze_hooks + 
        pred_toks = output.logits[:,-1,:].argmax(dim=-1)
        tokens["input_ids"] = torch.cat([tokens["input_ids"], pred_toks.unsqueeze(-1)], dim=1)
        del output
    
    for j, (_, row) in enumerate(batch_rows.iterrows()):
        generated_text = tokenizer.decode(tokens["input_ids"][j,input_length:]).replace(tokenizer.pad_token[-1], "")
        _config.write_to_csv(filepath, row.to_list() + [generated_text, -1])

    del tokens, hook
    torch.cuda.empty_cache()
    gc.collect()


  0%|                                                                                          | 0/11 [00:00<?, ?it/s]

100%|█████████████████████████████████████████████████████████████████████████████████| 11/11 [04:17<00:00, 23.45s/it]
